# SC24 Informational Analysis — Solar Cycle 24

Analysis code for: **"Informational analysis of TSA and PSI in Solar Cycle 24"**  
Ferreira et al. (2026) — submitted to *Advances in Space Research*

Produces all 7 manuscript figures and reference data tables in a **single linear execution**.

---
**Run cells top to bottom without skipping any.**

## Section 0 — Setup

Imports, supermongo style, Courier Prime font.

In [ ]:
# CELL 0 — Imports
import numpy as np
import pandas as pd
import matplotlib
from astroquery.vizier import Vizier
from scipy.spatial.distance import jensenshannon   # adicionado F8b

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
from astropy.time import Time
from sklearn.metrics import mutual_info_score
from scipy.stats import mannwhitneyu, pearsonr
from kneed import KneeLocator
import warnings, math, os
warnings.filterwarnings('ignore')
print('All imports OK')

In [ ]:
# CELL 1 — Supermongo style + Courier Prime (no hard-coded paths)
def setup_style():
    for fp in fm.findSystemFonts():
        if 'courier' in fp.lower():
            try: fm.fontManager.addfont(fp)
            except Exception: pass
    plt.rcParams.update({
        'font.family':         'monospace',
        'font.monospace':      ['Courier Prime', 'Courier New', 'Courier'],
        'axes.grid':           False,
        'text.usetex':         False,
        'xtick.direction':     'in',
        'ytick.direction':     'in',
        'xtick.top':           True,
        'ytick.right':         True,
        'xtick.minor.visible': True,
        'ytick.minor.visible': True,
        'figure.dpi':          150,
        'savefig.dpi':         300,
        'savefig.bbox':        'tight',
    })

def sm_style(ax):
    """Apply supermongo-style ticks to an Axes."""
    ax.tick_params(axis='both', which='major', direction='in',
                   top=True, right=True, length=6, width=1.0)
    ax.tick_params(axis='both', which='minor', direction='in',
                   top=True, right=True, length=3, width=0.8)
    ax.minorticks_on()

setup_style()
print('Supermongo style active')

## Section 1 — Data Loading and Preprocessing

Loads Mandal et al. (2020), filters SC24, renames `Areac` -> `TSA`,  
verifies integrity against T0.1 audit (3636 obs, 89.8% coverage).

In [ ]:
# CELL 2 — Data loading and integrity verification
# For a local table (remove the comment):
#df = pd.read_csv('Mandall2020_catalog1.tsv', sep=';', comment='#',
#                 usecols=['Obs.date', 'Areac', 'PSI'])

#For a vizier query:
from astroquery.vizier import Vizier
v = Vizier(columns=['Obs.date', 'Areac', 'PSI'], row_limit=-1)
result = v.get_catalogs('J/A+A/640/A78')
df = result[0].to_pandas()

df['Obs.date'] = pd.to_datetime(df['Obs.date'])
df.set_index('Obs.date', inplace=True)
df.sort_index(inplace=True)
df.dropna(subset=['Areac', 'PSI'], inplace=True)

df_sc24 = df.loc['2008-12-01':'2019-12-31'].copy()
df_sc24 = df_sc24.rename(columns={'Areac': 'TSA'})  # canonical name

# Integrity assertions (T0_1_CONCLUIDA.md)
assert len(df_sc24) == 3636, f'Expected 3636, got {len(df_sc24)}'
assert df_sc24.index.max() == pd.Timestamp('2019-10-02')
assert df_sc24.index.min() == pd.Timestamp('2008-12-01')

print(f'Catalogue: {len(df_sc24)} obs ({len(df_sc24)/4048*100:.1f}% coverage)')
print(f'Period: {df_sc24.index.min().date()} to {df_sc24.index.max().date()}')
print(f'TSA=0: {(df_sc24["TSA"]==0).sum()} days ({(df_sc24["TSA"]==0).sum()/len(df_sc24)*100:.1f}%)')

In [ ]:
# CELL 3 — Normalisation (max-abs) and temporal aggregation
# Creates df_mensal_avg and df_anual_avg explicitly
# (resolves hidden dependency from geral.ipynb)
df_normalizado = df_sc24.copy()
for col in ['TSA', 'PSI']:
    df_normalizado[col] = df_sc24[col] / df_sc24[col].max()

df_mensal_avg = df_normalizado.resample('ME').mean()
df_anual_avg  = df_normalizado.resample('YE').mean()

print(f'Daily: {len(df_normalizado)}  Monthly: {len(df_mensal_avg)}  Annual: {len(df_anual_avg)}')

## Section 2 — Information Theory Functions

Shannon H, Lempel-Ziv LZC, permutation entropy PE, mutual information MI.

In [ ]:
# CELL 4 — Information theory functions
def shannon_entropy(series):
    """Shannon entropy in bits."""
    counts = pd.Series(series).value_counts()
    probs  = counts / len(series)
    return float(-np.sum(probs * np.log2(probs + 1e-12)))

def lempel_ziv_complexity(series):
    """Normalised Lempel-Ziv complexity (Lempel & Ziv 1976)."""
    s = ''.join(map(str, np.array(series, dtype=int)))
    n = len(s)
    if n == 0: return 0.0
    dictionary, i, c = set(), 0, 0
    while i < n:
        j = i
        while j < n and s[i:j+1] in dictionary: j += 1
        dictionary.add(s[i:j+1]); c += 1; i = j + 1
    return c / n

def permutation_entropy(series, m=3, normalize=True):
    """Permutation entropy (Bandt & Pompe 2002)."""
    x = np.array(series); n = len(x)
    if n < m: return np.nan
    patterns = {}
    for i in range(n - m + 1):
        perm = tuple(np.argsort(x[i:i+m]))
        patterns[perm] = patterns.get(perm, 0) + 1
    total = sum(patterns.values())
    probs = np.array(list(patterns.values())) / total
    pe = -np.sum(probs * np.log2(probs + 1e-12))
    if normalize:
        max_pe = np.log2(math.factorial(m))
        if max_pe > 0: pe /= max_pe
    return float(pe)

def mutual_information(s1, s2):
    """Mutual information in bits."""
    return float(mutual_info_score(s1, s2) / np.log(2))

print('Functions loaded: shannon_entropy, lempel_ziv_complexity, permutation_entropy, mutual_information')

## Section 3 — Discretisation and Static Metrics (Table II)

**T1.1 FIX:** `pd.cut()` (equal-width bins) replaces `pd.qcut()` (bug).  
Original `qcut` forced H = log2(10) = 3.3219 bits by construction.

In [ ]:
# CELL 5 — Discretisation: pd.cut() — CORRECTED (NOT qcut)
# pd.qcut(): equal-frequency => p_i=1/N_BINS => H=log2(N_BINS) always (BUG)
# pd.cut():  equal-width    => H reflects actual distribution (CORRECT)
N_BINS = 10

df_diario_disc = pd.DataFrame(index=df_normalizado.index)
df_mensal_disc = pd.DataFrame(index=df_mensal_avg.index)
df_anual_disc  = pd.DataFrame(index=df_anual_avg.index)

for col in ['TSA', 'PSI']:
    df_diario_disc[col] = pd.cut(df_normalizado[col],  N_BINS, labels=False, duplicates='drop')
    df_mensal_disc[col] = pd.cut(df_mensal_avg[col],   N_BINS, labels=False, duplicates='drop')
    df_anual_disc[col]  = pd.cut(df_anual_avg[col],    N_BINS, labels=False, duplicates='drop')

for df_d in [df_diario_disc, df_mensal_disc, df_anual_disc]:
    df_d.dropna(inplace=True)

# Anti-regression: H must differ from log2(10)
h_test = shannon_entropy(df_diario_disc['TSA'])
assert abs(h_test - np.log2(10)) > 0.05, f'FATAL qcut: H={h_test:.4f}'
print(f'H(TSA) daily = {h_test:.4f} bits  !=  {np.log2(10):.4f}  [log2(10)]  OK')

In [ ]:
# CELL 6 — Table II: static information-theory metrics
# PE: m=4 for daily, m=3 for monthly and annual (SP2026 convention)
# H, LZC, I  -> computed on DISCRETISED data (df_*_disc): correct, symbol-based.
# PE          -> computed on CONTINUOUS data (df_*_cont): PE is an ordinal-pattern
#                metric and REQUIRES continuous-valued input. Feeding discretised
#                (10-level) data creates massive ties that np.argsort breaks by
#                array position, fabricating spurious ordinal patterns. [FIX F-PE]
datasets_static = {
    'Daily':   (df_diario_disc, df_normalizado),
    'Monthly': (df_mensal_disc, df_mensal_avg),
    'Annual':  (df_anual_disc,  df_anual_avg),
}
results_static  = {}

for scale, (df_d, df_c) in datasets_static.items():
    m_pe = 4 if scale == 'Daily' else 3
    results_static[scale] = {
        'H(TSA)':     shannon_entropy(df_d['TSA']),
        'H(PSI)':     shannon_entropy(df_d['PSI']),
        'PE(TSA)':    permutation_entropy(df_c['TSA'], m=m_pe),
        'PE(PSI)':    permutation_entropy(df_c['PSI'], m=m_pe),
        'LZC(TSA)':   lempel_ziv_complexity(df_d['TSA']),
        'LZC(PSI)':   lempel_ziv_complexity(df_d['PSI']),
        'I(TSA;PSI)': mutual_information(df_d['TSA'], df_d['PSI']),
    }

df_table_II = pd.DataFrame(results_static).T
print('=== TABLE II ===')
print(df_table_II.round(4).to_string())
df_table_II.to_csv('table_II.csv')
print('Saved: table_II.csv')

## Section 4 — Bin Number Stability Analysis (Fig. 2)

**T1.1 second fix:** `pd.cut()` in stability loop too (original Cell 50 of geral.ipynb).

In [ ]:
# CELL 7 — Bin stability loop (pd.cut — second qcut correction)
n_daily   = len(df_normalizado)
n_monthly = len(df_mensal_avg.dropna())
sturges_daily   = int(1 + np.log2(n_daily))
sturges_monthly = int(1 + np.log2(n_monthly))
print(f'Sturges daily (N={n_daily}): k={sturges_daily}')
print(f'Sturges monthly (N={n_monthly}): k={sturges_monthly}')

stab_daily, stab_monthly = [], []
for n in range(3, 26):
    try:
        td = pd.cut(df_normalizado['TSA'], n, labels=False, duplicates='drop')
        pd_ = pd.cut(df_normalizado['PSI'], n, labels=False, duplicates='drop')
        m = td.notna() & pd_.notna()
        stab_daily.append({'n_bins': n, 'MI': mutual_information(td[m], pd_[m])})
    except Exception: pass
    try:
        tm = pd.cut(df_mensal_avg['TSA'], n, labels=False, duplicates='drop')
        pm = pd.cut(df_mensal_avg['PSI'], n, labels=False, duplicates='drop')
        m = tm.notna() & pm.notna()
        stab_monthly.append({'n_bins': n, 'MI': mutual_information(tm[m], pm[m])})
    except Exception: pass

df_stab_daily   = pd.DataFrame(stab_daily).set_index('n_bins')
df_stab_monthly = pd.DataFrame(stab_monthly).set_index('n_bins')
print(f'MI(n=10) daily = {df_stab_daily.loc[10,"MI"]:.4f} bits')

In [ ]:
# CELL 8 — Figure 2: bins stability -> 2.png
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(df_stab_daily.index,   df_stab_daily['MI'],   'o-',  color='#1f4e79', lw=1.5, ms=4, label='Daily')
ax.plot(df_stab_monthly.index, df_stab_monthly['MI'], 's--', color='#c55a11', lw=1.5, ms=4, label='Monthly')
ax.axvline(x=N_BINS, color='black', ls=':', lw=1.2, label=f'Chosen n={N_BINS}')
ax.axvline(x=sturges_daily,   color='#1f4e79', ls='--', lw=0.8, alpha=0.6, label=f'Sturges daily (k={sturges_daily})')
ax.axvline(x=sturges_monthly, color='#c55a11', ls='--', lw=0.8, alpha=0.6, label=f'Sturges monthly (k={sturges_monthly})')
ax.set_xlabel('Number of bins', fontsize=12)
ax.set_ylabel('I(TSA; PSI)  (bits)', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
sm_style(ax)
plt.tight_layout()
plt.savefig('2.png')
plt.show()
print('2.png saved')

## Section 5 — Optimal Window Stability (Fig. 3)

In [ ]:
# CELL 9 — Window stability search
dados_m = df_mensal_disc.dropna()
busca   = []
for tam in range(6, 61, 2):
    mis = [mutual_information(dados_m.iloc[s:s+tam]['TSA'], dados_m.iloc[s:s+tam]['PSI'])
           for s in range(0, len(dados_m)-tam)]
    if mis:
        busca.append({'WindowSize': tam, 'MI_std': np.std(mis)})

df_busca = pd.DataFrame(busca).set_index('WindowSize')
try:
    k = KneeLocator(df_busca.index.tolist(), df_busca['MI_std'].tolist(),
                    S=1.0, curve='convex', direction='decreasing')
    cotovelo = k.elbow if k.elbow else 26
except Exception: cotovelo = 26

WINDOW = 26  # article-confirmed optimal for SC24
print(f'KneeLocator elbow: {cotovelo} months | Article window (WINDOW): {WINDOW} months')

### ~~CELL 10 — Fig. 3 old (fig3_window_stability.png)~~

**DEPRECATED** — Replaced by **CELL F9.1** which saves `3.png` with QBO label and σ annotation.

Original code preserved as comment below for reproducibility.


## Section 6 — Dynamic Analysis with 26-month Window (Fig. 4)

**T1.2:** Bootstrap 95% CI (n_boot=500) on H, LZC, MI.  
**T1.3:** Permutation entropy (m=3) in every window.  
Output: `df_resultados_janela_v2.csv`

## Section C_LMC — Statistical Complexity (López-Ruiz 1995)

C_LMC = H_norm × D_JS  where D_JS is the Jensen-Shannon divergence
between the observed distribution and the uniform (maximum-entropy) reference.
Computed for each 26-month sliding window for both TSA and PSI.
Functions defined here; called inside the sliding-window loop (Cell 11).

In [ ]:
# CELL C1 — C_LMC functions (F8b)

def lmc_complexity(p, N_bins=10):
    """Complexidade estatística LMC (López-Ruiz 1995).
    C_LMC = H_norm * D_JS
    D_JS = Jensen-Shannon divergence entre p e distribuição uniforme.
    Retorna: (C_lmc, H_norm, D_js)
    """
    p = np.array(p, dtype=float)
    p = p + 1e-15
    p = p / p.sum()
    p_uniform = np.ones(N_bins) / N_bins
    H      = -np.sum(p * np.log2(p))
    H_max  = np.log2(N_bins)
    H_norm = H / H_max
    D_js   = jensenshannon(p, p_uniform, base=2) ** 2
    C_lmc  = H_norm * D_js
    return float(C_lmc), float(H_norm), float(D_js)


def lmc_upper_bound(N_bins=10, n_points=1000):
    """Curva C_max(H_norm) para N_bins (varredura analítica).
    Para cada k=1..N_bins: distribui probabilidade em k bins equiprováveis.
    Retorna: (H_vals, C_vals) — arrays para o plano C–H (Fig. 9).
    """
    H_vals, C_vals = [], []
    for k in range(1, N_bins + 1):
        for frac in np.linspace(1/k, 1.0, n_points // N_bins):
            p = np.zeros(N_bins)
            p[:k] = frac / k
            residual = 1.0 - p[:k].sum()
            if residual > 0:
                p[0] += residual
            p = np.clip(p, 0, None)
            if p.sum() > 0:
                C, H, _ = lmc_complexity(p, N_bins)
                H_vals.append(H)
                C_vals.append(C)
    return np.array(H_vals), np.array(C_vals)


# Pré-calcular curvas teóricas (usadas em F9 para Fig. 9)
H_curve, C_curve = lmc_upper_bound(N_bins=N_BINS)
print(f'✓ lmc_complexity e lmc_upper_bound definidas.')
print(f'  Curva teórica: {len(H_curve)} pontos  H∈[{H_curve.min():.3f}, {H_curve.max():.3f}]')

In [ ]:
# CELL 11 — Sliding window + bootstrap CI + PE + C_LMC  (F8b)
# Runtime: ~1-2 min (106 windows × 500 bootstrap samples)
N_BOOT          = 500
results_dynamic = []
data_monthly    = df_mensal_disc.dropna()
print(f'Windows: {len(data_monthly)-WINDOW+1}  Bootstrap samples: {N_BOOT}')

for start in range(0, len(data_monthly) - WINDOW + 1):
    window = data_monthly.iloc[start:start + WINDOW]
    ts     = window.index[WINDOW // 2]
    # continuous counterpart of this window (same dates) for PE [FIX F-PE]
    window_cont = df_mensal_avg.loc[window.index]

    # — métricas existentes —————————————————————————————————————
    h_tsa   = shannon_entropy(window['TSA'])
    h_psi   = shannon_entropy(window['PSI'])
    lzc_tsa = lempel_ziv_complexity(window['TSA'])
    lzc_psi = lempel_ziv_complexity(window['PSI'])
    mi_val  = mutual_information(window['TSA'], window['PSI'])
    pe_tsa  = permutation_entropy(window_cont['TSA'], m=3)  # continuous input [FIX F-PE]
    pe_psi  = permutation_entropy(window_cont['PSI'], m=3)  # continuous input [FIX F-PE]

    # — vetores de probabilidade para C_LMC (F8b) ———————————————
    bins_tsa = pd.cut(window['TSA'], N_BINS, labels=False).value_counts().sort_index()
    bins_psi = pd.cut(window['PSI'], N_BINS, labels=False).value_counts().sort_index()
    # garantir N_BINS entradas (bins vazios = 0)
    p_tsa = np.zeros(N_BINS); p_psi = np.zeros(N_BINS)
    for b, v in bins_tsa.items():
        if pd.notna(b): p_tsa[int(b)] = v
    for b, v in bins_psi.items():
        if pd.notna(b): p_psi[int(b)] = v
    p_tsa = p_tsa / p_tsa.sum() if p_tsa.sum() > 0 else np.ones(N_BINS)/N_BINS
    p_psi = p_psi / p_psi.sum() if p_psi.sum() > 0 else np.ones(N_BINS)/N_BINS

    c_lmc_tsa, h_norm_tsa, d_js_tsa = lmc_complexity(p_tsa, N_BINS)
    c_lmc_psi, h_norm_psi, d_js_psi = lmc_complexity(p_psi, N_BINS)

    # — bootstrap (métricas existentes) ——————————————————————————
    bh, bhp, blzc, blzcp, bmi = [], [], [], [], []
    idx = np.arange(WINDOW)
    for _ in range(N_BOOT):
        s = np.random.choice(idx, WINDOW, replace=True)
        w = window.iloc[s]
        bh.append(shannon_entropy(w['TSA']))
        bhp.append(shannon_entropy(w['PSI']))
        blzc.append(lempel_ziv_complexity(w['TSA']))
        blzcp.append(lempel_ziv_complexity(w['PSI']))
        bmi.append(mutual_information(w['TSA'], w['PSI']))

    results_dynamic.append({
        # — 18 colunas originais (inalteradas) ————————————————————
        'Date': ts,
        'H_TSA': h_tsa,      'H_PSI': h_psi,
        'H_TSA_lo': np.percentile(bh,   2.5),  'H_TSA_hi': np.percentile(bh,   97.5),
        'H_PSI_lo': np.percentile(bhp,  2.5),  'H_PSI_hi': np.percentile(bhp,  97.5),
        'LZC_TSA': lzc_tsa,  'LZC_PSI': lzc_psi,
        'LZC_TSA_lo': np.percentile(blzc,  2.5), 'LZC_TSA_hi': np.percentile(blzc,  97.5),
        'LZC_PSI_lo': np.percentile(blzcp, 2.5), 'LZC_PSI_hi': np.percentile(blzcp, 97.5),
        'MI': mi_val,
        'MI_lo': np.percentile(bmi, 2.5), 'MI_hi': np.percentile(bmi, 97.5),
        'PE_TSA': pe_tsa,    'PE_PSI': pe_psi,
        # — 6 colunas novas F8b ————————————————————————————————————
        'C_LMC_TSA': c_lmc_tsa, 'C_LMC_PSI': c_lmc_psi,
        'H_norm_TSA': h_norm_tsa, 'H_norm_PSI': h_norm_psi,
        'D_JS_TSA': d_js_tsa,    'D_JS_PSI': d_js_psi,
    })

df_dynamic = pd.DataFrame(results_dynamic).set_index('Date')
df_dynamic.to_csv('df_resultados_janela_v2.csv')

# Verificações de integridade
assert df_dynamic.shape[1] == 23, f"Esperado 23 colunas (sem Date), encontrado {df_dynamic.shape[1]}"
assert df_dynamic['H_TSA'].max() < 3.32, "ERRO: H_TSA próximo de log2(10) — checar qcut!"
print(f'{len(df_dynamic)} windows computed. Colunas: {df_dynamic.shape[1]+1} (Date+23).')
print(f'Saved: df_resultados_janela_v2.csv')
print(f'\nC_LMC_TSA → max={df_dynamic["C_LMC_TSA"].max():.4f} em {df_dynamic["C_LMC_TSA"].idxmax().date()}')
print(f'            min={df_dynamic["C_LMC_TSA"].min():.4f} em {df_dynamic["C_LMC_TSA"].idxmin().date()}')
print(f'C_LMC_PSI → max={df_dynamic["C_LMC_PSI"].max():.4f} em {df_dynamic["C_LMC_PSI"].idxmax().date()}')

In [ ]:
# CELL FIG4_SHADE — Fig. 4: 4 panels (H, LZC, I, PE) + phase shading
# Adds: axvspan by phase, vertical dashed lines, Gnevyshev gap marker
# Data: df_dynamic (already in memory)
# Output: 4.png | DPI 300

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd

_avail = {f.name for f in fm.fontManager.ttflist}
MONO = next((f for f in ['Courier Prime','Courier New','DejaVu Sans Mono']
             if f in _avail), 'monospace')
plt.rcParams.update({'font.family':'monospace','font.monospace':[MONO],'font.size':9})

df = df_dynamic.copy()
_mi_peak = df['MI'].idxmax()
_c_peak  = df['C_LMC_TSA'].idxmax()
_gd      = pd.Timestamp('2012-08-01')   # Gnevyshev gap
_gd_near = df.index[np.argmin(np.abs(df.index - _gd))]

_COL_ASC  = '#2171B5'
_COL_MAX  = '#FF6F00'
_COL_DESC = '#D84315'
_ALPHA    = 0.09

def _shade(ax):
    ax.axvspan(df.index.min(), _mi_peak,       alpha=_ALPHA, color=_COL_ASC,  zorder=0)
    ax.axvspan(_mi_peak,       _c_peak,          alpha=_ALPHA, color=_COL_MAX,  zorder=0)
    ax.axvspan(_c_peak,        df.index.max(),   alpha=_ALPHA, color=_COL_DESC, zorder=0)
    ax.axvline(_mi_peak, color='#AAAAAA', lw=0.7, ls='--', zorder=1)
    ax.axvline(_c_peak,  color='#AAAAAA', lw=0.7, ls='--', zorder=1)
    ax.axvline(_gd_near, color='#9C27B0', lw=0.7, ls=':',  zorder=1, alpha=0.6)

fig, axes = plt.subplots(4, 1, figsize=(9.0, 10.0), dpi=300,
                          sharex=True,
                          gridspec_kw={'hspace': 0.06})

# ── Panel 1: H ────────────────────────────────────────────────────────────────
ax = axes[0]; _shade(ax)
ax.plot(df.index, df['H_TSA'], color='#2171B5', lw=1.6, label='H(TSA)')
ax.plot(df.index, df['H_PSI'], color='#E6550D', lw=1.4, ls='--',
        alpha=0.75, label='H(PSI)')
ax.fill_between(df.index, df['H_TSA_lo'], df['H_TSA_hi'],
                color='#2171B5', alpha=0.15)
ax.fill_between(df.index, df['H_PSI_lo'], df['H_PSI_hi'],
                color='#E6550D', alpha=0.12)
ax.set_ylabel('H (bits)', fontsize=9)
ax.legend(fontsize=8, loc='upper right', prop={'family':MONO,'size':8})
ax.tick_params(labelsize=8)

# Phase labels above Panel 1
for _t0, _t1, _lbl, _col in [
    (df.index.min(), _mi_peak,      'Ascending',   _COL_ASC),
    (_mi_peak,       _c_peak,        'Maximum',     _COL_MAX),
    (_c_peak,        df.index.max(), 'Descending',  _COL_DESC),
]:
    _tm = _t0 + (_t1 - _t0) / 2
    ax.text(_tm, 1.01, _lbl,
        transform=ax.get_xaxis_transform(),
        ha='center', va='bottom', fontsize=7.5,
        color=_col, fontweight='bold', clip_on=False)

# ── Panel 2: LZC ──────────────────────────────────────────────────────────────
ax = axes[1]; _shade(ax)
ax.plot(df.index, df['LZC_TSA'], color='#2171B5', lw=1.6, label='LZC(TSA)')
ax.plot(df.index, df['LZC_PSI'], color='#E6550D', lw=1.4, ls='--',
        alpha=0.75, label='LZC(PSI)')
ax.fill_between(df.index, df['LZC_TSA_lo'], df['LZC_TSA_hi'],
                color='#2171B5', alpha=0.15)
ax.fill_between(df.index, df['LZC_PSI_lo'], df['LZC_PSI_hi'],
                color='#E6550D', alpha=0.12)
ax.set_ylabel('LZC', fontsize=9)
ax.legend(fontsize=8, loc='upper right', prop={'family':MONO,'size':8})
ax.tick_params(labelsize=8)

# ── Panel 3: I ────────────────────────────────────────────────────────────────
ax = axes[2]; _shade(ax)
ax.plot(df.index, df['MI'], color='#2A7D2E', lw=1.6, label='I(TSA; PSI)')
ax.fill_between(df.index, df['MI_lo'], df['MI_hi'],
                color='#2A7D2E', alpha=0.15)
# Gnevyshev gap annotation
ax.annotate('G-dip',
            xy=(_gd_near, df.loc[_gd_near, 'MI']),
            xytext=(_gd_near + pd.DateOffset(months=8),
                    df.loc[_gd_near, 'MI'] - 0.30),
            fontsize=7.5, color='#9C27B0',
            arrowprops=dict(arrowstyle='->', color='#9C27B0', lw=0.8))
ax.set_ylabel('I (bits)', fontsize=9)
ax.legend(fontsize=8, loc='upper right', prop={'family':MONO,'size':8})
ax.tick_params(labelsize=8)

# ── Panel 4: PE ───────────────────────────────────────────────────────────────
ax = axes[3]; _shade(ax)
ax.plot(df.index, df['PE_TSA'], color='#2171B5', lw=1.6, label='PE(TSA)')
ax.plot(df.index, df['PE_PSI'], color='#E6550D', lw=1.4, ls='--',
        alpha=0.75, label='PE(PSI)')
ax.set_ylabel('PE (norm.)', fontsize=9)
ax.set_xlabel('Year', fontsize=9)
ax.legend(fontsize=8, loc='upper right', prop={'family':MONO,'size':8})
ax.tick_params(labelsize=8)

# ── x-axis ────────────────────────────────────────────────────────────────────
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].set_xlim(df.index.min(), df.index.max())

for ax in axes:
    for sp in ax.spines.values(): sp.set_linewidth(0.6)
    ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.savefig('4.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('4.png saved')

## Section 7 — Quantitative Comparison: Minimum vs. Maximum

In [ ]:
# CELL 13 — Mann-Whitney U (preserves logic of geral.ipynb Cell 36)
df_act = df_normalizado.resample('ME').mean()
act    = df_act['TSA']
LIM_MIN, LIM_MAX = act.quantile(0.33), act.quantile(0.66)

def classify(a):
    if a <= LIM_MIN: return 'Minimum'
    elif a >= LIM_MAX: return 'Maximum'
    return 'Transition'

phases = act.apply(classify).to_frame('Phase')
dyn2 = df_dynamic.copy()
dyn2['mes_ano'] = dyn2.index.to_period('M')
phases['mes_ano'] = phases.index.to_period('M')
merged = pd.merge(dyn2.reset_index(), phases.reset_index()[['mes_ano','Phase']],
                  on='mes_ano', how='left').set_index('Date')

comparison = {}
for m in ['H_TSA','H_PSI','LZC_TSA','LZC_PSI','MI','PE_TSA','PE_PSI']:
    a = merged[merged['Phase']=='Minimum'][m].dropna()
    b = merged[merged['Phase']=='Maximum'][m].dropna()
    if len(a)>1 and len(b)>1:
        stat, pval = mannwhitneyu(a, b, alternative='two-sided')
        comparison[m] = {'Min_mean':a.mean(),'Min_std':a.std(),
                         'Max_mean':b.mean(),'Max_std':b.std(),
                         'MW_U':stat,'p_value':pval}

df_comparison = pd.DataFrame(comparison).T
df_comparison.to_csv('table_min_max.csv')
print('=== Min vs. Max (Mann-Whitney U) ===')
print(df_comparison[['Min_mean','Max_mean','p_value']].round(4).to_string())

In [ ]:
# CELL KS — KS test ascending vs. descending branches (required by CELL 18)
# Computes: ks_D_MI, ks_p_MI, ks_D_LZC, ks_p_LZC, ks_D_PE, ks_p_PE
# Loop direction and closure gap also computed here.
# These variables are used by CELL 18 (tabela_referencia_v2.csv)
# and by CELL FIG6_NEW (KS box annotation).

from scipy.stats import ks_2samp
import numpy as np

# Branch split at the OFFICIAL SC24 maximum (April 2014, 13-month smoothed SSN
# = 81.8; SIDC/SILSO news004). 2014-06-01 sits at this maximum within the
# smoothing window and cleanly separates ascending from descending branch.
# NOTE: the MI peak (Oct 2014) lags the SSN maximum; this lag is itself a result.
SC24_MAX_KS = pd.Timestamp('2014-06-01')
_ks_asc  = df_dynamic[df_dynamic.index <= SC24_MAX_KS]
_ks_desc = df_dynamic[df_dynamic.index >  SC24_MAX_KS]

# KS test: I(TSA;PSI)
ks_D_MI,  ks_p_MI  = ks_2samp(_ks_asc['MI'].dropna(),
                                _ks_desc['MI'].dropna())

# KS test: LZC(TSA)
ks_D_LZC, ks_p_LZC = ks_2samp(_ks_asc['LZC_TSA'].dropna(),
                                _ks_desc['LZC_TSA'].dropna())

# KS test: PE(TSA)
ks_D_PE,  ks_p_PE  = ks_2samp(_ks_asc['PE_TSA'].dropna(),
                                _ks_desc['PE_TSA'].dropna())

# Loop direction (signed polygon area — CW if negative)
_lx = df_dynamic['MI'].values
_ly = df_dynamic['LZC_TSA'].values
_loop_signed_area = 0.5 * np.sum(_lx[:-1]*_ly[1:] - _lx[1:]*_ly[:-1])
loop_direction = 'CW' if _loop_signed_area < 0 else 'CCW'

# Closure gap (Euclidean distance start→end / max extent)
_p_start = np.array([df_dynamic['MI'].iloc[0],  df_dynamic['LZC_TSA'].iloc[0]])
_p_end   = np.array([df_dynamic['MI'].iloc[-1], df_dynamic['LZC_TSA'].iloc[-1]])
_extent  = np.sqrt((df_dynamic['MI'].max()-df_dynamic['MI'].min())**2 +
                   (df_dynamic['LZC_TSA'].max()-df_dynamic['LZC_TSA'].min())**2)
loop_closure_gap_pct = np.linalg.norm(_p_end - _p_start) / _extent * 100  # 2D def (i) [CANONICAL]

# Reference-only: 1D MI-axis closure gap (NOT used in manuscript; kept for audit)
_gap_mi_axis = abs(df_dynamic['MI'].iloc[-1] - df_dynamic['MI'].iloc[0]) \
               / df_dynamic['MI'].max() * 100
print(f'[audit] closure gap 1D MI-axis = {_gap_mi_axis:.1f}% (informative only; 2D is canonical)')

# Aliases expected by CELL 19
loop_dir = loop_direction
gap_pct  = round(loop_closure_gap_pct, 1)

print(f'KS test (ascending vs. descending):')
print(f'  I(TSA;PSI): D = {ks_D_MI:.4f}, p = {ks_p_MI:.4f}')
print(f'  LZC(TSA):   D = {ks_D_LZC:.4f}, p = {ks_p_LZC:.4f}')
print(f'  PE(TSA):    D = {ks_D_PE:.4f}, p = {ks_p_PE:.4f}')
print(f'Loop direction: {loop_direction}  |  Closure gap: {loop_closure_gap_pct:.1f}%')
print(f'N ascending = {len(_ks_asc)}  |  N descending = {len(_ks_desc)}')

## Section 8 — Static Analysis Figures

In [ ]:
# CELL 14 — Figure 1: time series 3 scales + JD axis -> 1.png
df_np = df_normalizado.copy()
df_np['JD'] = Time(df_np.index.to_numpy()).jd - 2440000

df_mp = df_normalizado.resample('ME').agg(
    TSA_mean=('TSA','mean'), TSA_std=('TSA','std'),
    PSI_mean=('PSI','mean'), PSI_std=('PSI','std'))
df_mp['JD'] = Time(df_mp.index.to_numpy()).jd - 2440000

df_ap = df_normalizado.resample('YE').agg(
    TSA_mean=('TSA','mean'), TSA_std=('TSA','std'),
    PSI_mean=('PSI','mean'), PSI_std=('PSI','std'))
df_ap['JD'] = Time(df_ap.index.to_numpy()).jd - 2440000

fig, axes = plt.subplots(3, 1, figsize=(10, 10), gridspec_kw={'hspace': 0.35})
for ax, (dpl, title) in zip(axes, [(df_np,'Daily'),(df_mp,'Monthly'),(df_ap,'Annual')]):
    if 'TSA_mean' in dpl.columns:
        ax.plot(dpl['JD'], dpl['TSA_mean'], color='#1f4e79', lw=1.2, label='TSA')
        ax.plot(dpl['JD'], dpl['PSI_mean'], color='#c55a11', lw=1.2, label='PSI')
    else:
        ax.plot(dpl['JD'], dpl['TSA'], color='#1f4e79', lw=0.6, alpha=0.5, label='TSA')
        ax.plot(dpl['JD'], dpl['PSI'], color='#c55a11', lw=0.6, alpha=0.5, label='PSI')
    ax.set_ylabel('Normalised value', fontsize=10)
    ax.set_title(title, fontsize=10, pad=3)
    ax.legend(fontsize=8, loc='upper right')
    sm_style(ax)
axes[2].set_xlabel('JD - 2 440 000', fontsize=10)
plt.savefig('1.png')
plt.show()
print('1.png saved')

In [ ]:
# CELL 15 — Figure 5: joint heatmap P(TSA,PSI) -> 5.png
tsa_d = df_diario_disc['TSA'].dropna().astype(int)
psi_d = df_diario_disc['PSI'].dropna().astype(int)
idx   = tsa_d.index.intersection(psi_d.index)
jf, xe, ye = np.histogram2d(tsa_d.loc[idx], psi_d.loc[idx], bins=N_BINS)
jp = jf / jf.sum()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(jp.T, origin='lower', aspect='auto', cmap='plasma',
               extent=[xe[0], xe[-1], ye[0], ye[-1]])
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('P(TSA, PSI)', fontsize=10)
ax.set_xlabel('TSA bin', fontsize=11)
ax.set_ylabel('PSI bin', fontsize=11)
sm_style(ax)
plt.tight_layout()
plt.savefig('5.png')
plt.show()
print('5.png saved')

## Section 9 — Informational Phase Spaces

**T1.5:** Surrogate test (N=1000) for Fig. 6.  
**T1.4:** Pearson r + regression line for Fig. 7.

### ~~CELL 16 — Fig. 6 old (LZC×I phase space)~~

**DEPRECATED** — Replaced by **CELL FIG6_NEW** which shows D_JS(t) + I(t) temporal panels (`6.png`).

Original code preserved as comment below for reproducibility.


In [ ]:
# CELL 17 — Figure 7: H vs LZC + Pearson r + regression -> 7.png
df_ph7  = df_dynamic[['H_TSA','LZC_TSA']].dropna()
r_tsa, p_tsa = pearsonr(df_ph7['H_TSA'], df_ph7['LZC_TSA'])
print(f'Pearson r(H_TSA, LZC_TSA): r={r_tsa:.4f}  p={p_tsa:.2e}')

m_reg, b_reg = np.polyfit(df_ph7['H_TSA'], df_ph7['LZC_TSA'], 1)
x_line = np.linspace(df_ph7['H_TSA'].min(), df_ph7['H_TSA'].max(), 100)
n_ph7  = len(df_ph7)

fig, ax = plt.subplots(figsize=(8, 7))
sc7 = ax.scatter(df_ph7['H_TSA'], df_ph7['LZC_TSA'], c=np.arange(n_ph7),
                 cmap='viridis', alpha=0.9, s=40, zorder=3)
ax.plot(x_line, m_reg*x_line+b_reg, 'k--', lw=1.2,
        label=f'Linear fit (r={r_tsa:.3f}, p={p_tsa:.1e})')
cbar7 = plt.colorbar(sc7, ax=ax)
cbar7.set_ticks([0, n_ph7-1])
cbar7.set_ticklabels([df_ph7.index[0].strftime('%Y-%m'), df_ph7.index[-1].strftime('%Y-%m')])
cbar7.set_label('Temporal evolution', rotation=270, labelpad=18, fontsize=10)
ax.annotate('Start/End\n(Minimum)', xy=(df_ph7['H_TSA'].iloc[0], df_ph7['LZC_TSA'].iloc[0]),
            xytext=(df_ph7['H_TSA'].iloc[0]-0.15, df_ph7['LZC_TSA'].iloc[0]+0.015),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
ax.annotate('Maximum', xy=(df_ph7['H_TSA'].iloc[n_ph7//2], df_ph7['LZC_TSA'].iloc[n_ph7//2]),
            xytext=(df_ph7['H_TSA'].iloc[n_ph7//2]+0.05, df_ph7['LZC_TSA'].iloc[n_ph7//2]-0.015),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
ax.set_xlabel('H(TSA)  (bits)', fontsize=11)
ax.set_ylabel('LZC(TSA)', fontsize=11)
ax.legend(fontsize=9, loc='upper left')
sm_style(ax)
plt.tight_layout()
plt.savefig('7.png')
plt.show()
print('7.png saved')

## Section F9 — Revised and New Figures

| Cell | Output | Manuscript figure |
|------|--------|------------------|
| CELL F9.1 | `3.png` | Fig. 3 — window stability (QBO label) |
| CELL FIG6_NEW | `6.png` | Fig. 6 — D_JS(t) + I(t) temporal |
| CELL FIG9_FIX | `9_CH.png` → `8.png` | Fig. 8 — C–H plane |
| CELL FIG10A | `9.png` | Fig. 9 — I(t) + C_LMC(t) hysteresis |

**Fig. 4** is produced by CELL FIG4_SHADE (Cell index 22).
**Fig. 8** (C_LMC vs time) was eliminated — content covered by Fig. 9 lower panel.


In [ ]:
# CELL F9.1 — Fig. 3 regenerated  [CORRECTED v3]
# FIX: both σ value AND percentage computed from df_dynamic (not hardcoded).
# Abstract will be updated in F10 to match these values.
# Output: 3.png | DPI 300

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

_avail = {f.name for f in fm.fontManager.ttflist}
MONO = next((f for f in ['Courier Prime', 'Courier New', 'DejaVu Sans Mono']
             if f in _avail), 'monospace')
plt.rcParams.update({'font.family': 'monospace',
                     'font.monospace': [MONO],
                     'font.size': 9})

# ── σ(I) stability curve (visual only) ───────────────────────────────────────
_sigma_W = {}
for _W in range(10, 61):
    _mi_vals = []
    for _s in range(0, len(df_mensal_disc) - _W + 1):
        _sl = df_mensal_disc.iloc[_s:_s + _W].dropna()
        if len(_sl) >= _W - 2:
            _mi_vals.append(
                mutual_information(_sl['TSA'].values, _sl['PSI'].values)
            )
    _sigma_W[_W] = float(np.std(_mi_vals)) if len(_mi_vals) > 1 else np.nan

_ws   = list(_sigma_W.keys())
_sigs = list(_sigma_W.values())

# ── CANONICAL VALUES: both derived from df_dynamic (single source of truth) ──
_sig26_canonical = float(df_dynamic['MI'].std())
_peak_canonical  = max(v for v in _sigs if not np.isnan(v))
_pct_canonical   = _sig26_canonical / _peak_canonical * 100

# Curve anchor point for arrow (visual only)
_sig26_curve = _sigma_W.get(26, _sig26_canonical)

print(f"σ(26m) = {_sig26_canonical:.4f} bits  |  peak = {_peak_canonical:.4f} bits")
print(f"% of peak = {_pct_canonical:.1f}%")
print(f"NOTE FOR F10: update abstract from 0.487/69.6% → "
      f"{_sig26_canonical:.3f}/{_pct_canonical:.1f}%")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7.0, 4.2), dpi=300)

ax.plot(_ws, _sigs, 'o-', color='#2171B5', lw=1.5, ms=4.5, zorder=3)
ax.axvline(26, color='#2171B5', ls='--', lw=1.4, zorder=2)


ax.text(27.2, _peak_canonical * 0.975,
        'Adopted window: 26 months\n(QBO ~24-28 m)',   # ASCII ~ instead of \u223c
        fontsize=8.5, va='top', color='#2171B5')

# Annotation — values from df_dynamic (consistent with notebook state)
ax.annotate(
    f'\u03c3(I) = {_sig26_canonical:.3f} bits\n({_pct_canonical:.1f}% of peak)',
    xy=(26, _sig26_curve),
    xytext=(36, _sig26_curve + (_peak_canonical - _sig26_curve) * 0.40),
    fontsize=8, color='#555555',
    arrowprops=dict(arrowstyle='->', color='#888888', lw=0.9)
)

ax.set_xlabel('Window size (months)')
ax.set_ylabel('\u03c3(I) (bits)')
ax.set_xlim(8, 62)
ax.tick_params(labelsize=8)
for sp in ax.spines.values():
    sp.set_linewidth(0.6)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.tight_layout(pad=0.8)
plt.savefig('3.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n3.png saved | σ = {_sig26_canonical:.3f} bits | {_pct_canonical:.1f}% of peak')

In [ ]:
# CELL FIG6_NEW — Fig. 6: D_JS(TSA) vs time + I(TSA;PSI) vs time
# Two aligned temporal panels with phase shading.
# D_JS_TSA already in df_dynamic (F8b). Phase colours consistent with Figs 9, 10A.
# KS test values cited in panel annotation.
# Output: 6.png | DPI 300

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

_avail = {f.name for f in fm.fontManager.ttflist}
MONO = next((f for f in ['Courier Prime','Courier New','DejaVu Sans Mono']
             if f in _avail), 'monospace')
plt.rcParams.update({'font.family':'monospace','font.monospace':[MONO],'font.size':9})

df = df_dynamic.copy()

# ── Phase boundaries ──────────────────────────────────────────────────────────
_mi_peak = df['MI'].idxmax()        # ascending → maximum split
_c_peak  = df['C_LMC_TSA'].idxmax() # maximum → descending split

_COL_ASC  = '#2171B5'
_COL_MAX  = '#FF6F00'
_COL_DESC = '#D84315'
_ALPHA_SHADE = 0.10

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 6.0), dpi=300,
                                sharex=True,
                                gridspec_kw={'hspace': 0.08})

def _add_phase_shading(ax):
    ax.axvspan(df.index.min(), _mi_peak,        alpha=_ALPHA_SHADE,
               color=_COL_ASC,  zorder=0)
    ax.axvspan(_mi_peak,        _c_peak,         alpha=_ALPHA_SHADE,
               color=_COL_MAX,  zorder=0)
    ax.axvspan(_c_peak,         df.index.max(),  alpha=_ALPHA_SHADE,
               color=_COL_DESC, zorder=0)
    ax.axvline(_mi_peak, color=_COL_ASC,  lw=0.8, ls='--', alpha=0.5, zorder=1)
    ax.axvline(_c_peak,  color=_COL_MAX,  lw=0.8, ls='--', alpha=0.5, zorder=1)

# ── Panel 1: D_JS(TSA) vs time ────────────────────────────────────────────────
_add_phase_shading(ax1)
ax1.plot(df.index, df['D_JS_TSA'], color=_COL_ASC, lw=1.8, zorder=3,
         label='$D_{JS}$(TSA)')
ax1.plot(df.index, df['D_JS_PSI'], color=_COL_DESC, lw=1.4, ls='--',
         alpha=0.65, zorder=3, label='$D_{JS}$(PSI)')
ax1.set_ylabel('$D_{JS}$ (bits$^{1/2}$)', fontsize=9)
ax1.legend(fontsize=8.5, loc='upper left', prop={'family': MONO, 'size': 8.5})
ax1.tick_params(labelsize=8)

# Phase labels (top panel)
for _t0, _t1, _lbl, _col in [
    (df.index.min(), _mi_peak,      'Ascending\n2010–2014', _COL_ASC),
    (_mi_peak,       _c_peak,        'Maximum\n2014–2016',  _COL_MAX),
    (_c_peak,        df.index.max(), 'Descending\n2016–2018',_COL_DESC),
]:
    _tm = _t0 + (_t1 - _t0) / 2
    ax1.text(_tm, 1.01, _lbl,
             transform=ax1.get_xaxis_transform(),
             ha='center', va='bottom', fontsize=7.5,
             color=_col, fontweight='bold', clip_on=False)

# KS annotation
try:
    _ks = (f'KS(I): D = {ks_D_MI:.3f}, p = {ks_p_MI:.4f}\n'
           f'KS(LZC): D = {ks_D_LZC:.3f}, p = {ks_p_LZC:.4f}')
except NameError:
    _ks = 'KS(I): D = 0.396, p = 0.0004\nKS(LZC): D = 0.358, p = 0.0020'
ax1.text(0.02, 0.04, _ks, transform=ax1.transAxes, fontsize=7.5,
         va='bottom', family=MONO,
         bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                   edgecolor='#CCCCCC', alpha=0.92))

# ── Panel 2: I(TSA;PSI) vs time ───────────────────────────────────────────────
_add_phase_shading(ax2)
ax2.plot(df.index, df['MI'], color='#2171B5', lw=1.8, zorder=3,
         label='$I$(TSA; PSI)')
ax2.fill_between(df.index, df['MI_lo'], df['MI_hi'],
                 color='#2171B5', alpha=0.18, zorder=2)

# MI peak annotation
ax2.annotate(
    f'I peak\n{_mi_peak.strftime("%b %Y")}\n{df.loc[_mi_peak,"MI"]:.3f} bits',
    xy=(_mi_peak, df.loc[_mi_peak, 'MI']),
    xytext=(_mi_peak - pd.DateOffset(months=20),
            df['MI'].max() * 0.82),
    fontsize=8, color='#0C447C', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#0C447C', lw=1.2),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
              edgecolor='#0C447C', alpha=0.92))


# Gnevyshev gap annotation
try:
    # [FIX 1D] true local MI minimum between the two sunspot peaks (2012-01..2013-06),
    # not nearest point to a fixed anchor (old logic gave 2012-07-31/1.845).
    _gap_slice = df.loc[pd.Timestamp('2012-01-01'):pd.Timestamp('2013-06-30'), 'MI']
    _gd_near = _gap_slice.idxmin()
    _gd_val  = df.loc[_gd_near, 'MI']
    ax2.annotate(
    f'Gnevyshev dip\n{_gd_near.strftime("%b %Y")}\n{_gd_val:.3f} bits',
    xy=(_gd_near, _gd_val),
    xytext=(_gd_near + pd.DateOffset(months=14),
            _gd_val - 0.55),
    fontsize=8, color='#6A1B9A', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=1.2),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
              edgecolor='#6A1B9A', alpha=0.92))
    print(f'  Gnevyshev dip annotated: {_gd_near.date()} = {_gd_val:.3f} bits')
except Exception as _e:
    print(f'  [warn] Gnevyshev annotation skipped: {_e}')

ax2.set_ylabel('$I$(TSA; PSI) (bits)', fontsize=9)
ax2.set_xlabel('Year', fontsize=9)
ax2.set_ylim(0, df['MI_hi'].max() * 1.06)
ax2.legend(fontsize=8.5, loc='upper left', prop={'family': MONO, 'size': 8.5})
ax2.tick_params(labelsize=8)

# ── Shared x-axis formatting ──────────────────────────────────────────────────
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.set_xlim(df.index.min(), df.index.max())

for ax in (ax1, ax2):
    for sp in ax.spines.values(): sp.set_linewidth(0.6)
    ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.savefig('6.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'6.png saved')
print(f'  D_JS_TSA range: [{df["D_JS_TSA"].min():.4f}, {df["D_JS_TSA"].max():.4f}]')
print(f'  Phase split: ascending→{_mi_peak.strftime("%Y-%m")}, '
      f'max→{_c_peak.strftime("%Y-%m")}')

In [ ]:
# CELL_2DD_FINAL — Fig. 8: 9_CH.png [FINAL]
# Fixes: y-axis zoom (0.06–0.28), label alternation in (a),
#        annotation z-order above curves, (a)(b) labels shifted up,
#        Det. chaos color legible for B&W print
# Output: 9_CH.png | DPI 300

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

_avail = {f.name for f in fm.fontManager.ttflist}
MONO = next((f for f in ['Courier Prime','Courier New','DejaVu Sans Mono']
             if f in _avail), 'monospace')
plt.rcParams.update({'font.family':'monospace','font.monospace':[MONO],'font.size':9})

df = df_dynamic.copy()
_h_peak_date = df['H_TSA'].idxmax()
_c_peak_date  = df['C_LMC_TSA'].idxmax()
_df_asc  = df[df.index <= _h_peak_date]
_df_desc = pd.concat([df.loc[[_h_peak_date]], df[df.index > _h_peak_date]])
_COL_ASC, _COL_DESC = '#2171B5', '#D84315'

# ── Y-axis range: zoom to data range ─────────────────────────────────────────
_Y_MIN = 0.06   # data min ~0.10 but C_max descends to ~0.06 at H_norm extremes
_Y_MAX = 0.285  # C_LMC max = 0.2371 + headroom for labels

# ── Theoretical C_max ─────────────────────────────────────────────────────────
try:
    _H_th,_C_th = lmc_upper_bound(N_bins=10,n_points=8000)
    _s=np.argsort(_H_th); _H_th,_C_th=_H_th[_s],_C_th[_s]
    _hb=np.linspace(0,1,600); _cm=np.full_like(_hb,np.nan)
    for _k,_h in enumerate(_hb):
        _m=np.abs(_H_th-_h)<0.012
        if _m.any(): _cm[_k]=_C_th[_m].max()
    _v=~np.isnan(_cm); _cs=np.full_like(_cm,np.nan)
    _cs[_v]=gaussian_filter1d(_cm[_v].astype(float),sigma=2.5)
    _has_c=True
except NameError: _has_c=False

# ── Smoothed ascending (w=4) ──────────────────────────────────────────────────
_W  = 4
_sa = _df_asc[['H_norm_TSA','C_LMC_TSA']].rolling(_W,center=True,min_periods=1).mean()

# ── Common axis setup ─────────────────────────────────────────────────────────
def _setup_ax(ax, ylabel=True):
    # Zones
    for _x0,_x1,_zcol,_lbl,_tc in [
        (0.0, 0.35,'#E6F1FB','Ordered', '#185FA5'),
        (0.35,0.72,'#EAF3DE','Complex', '#3B6D11'),
        (0.72,1.08,'#FAEEDA','Noisy',   '#854F0B'),
    ]:
        ax.axvspan(_x0,_x1,alpha=0.22,color=_zcol,zorder=0)
        ax.text((_x0+_x1)/2, _Y_MAX*0.98, _lbl,
                ha='center', fontsize=8, color=_tc,
                fontweight='bold', va='top', zorder=1)

    # C_max
    if _has_c:
        _v=~np.isnan(_cs)
        ax.plot(_hb[_v],_cs[_v],'-',color='#AAAAAA',lw=1.8,
                zorder=2, label='$C_{max}$ (upper bound)')

    # Reference markers — BELOW data (zorder=2)
    ax.scatter(1.0, 0.0, marker='s', s=70, fc='none', ec='#444',
           lw=1.5, zorder=8, clip_on=False)
    ax.text(0.97, _Y_MIN+0.004, 'White noise',
        fontsize=7.5, ha='right', color='#444', fontweight='bold',
        zorder=9,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor='#444', alpha=0.90))
    ax.scatter(0.42, 0.22, marker='^', s=70, fc='none', ec='#444',
           lw=1.5, zorder=8)
    ax.text(0.44, 0.221, 'Det. chaos\n(Rosso 2007)',
        fontsize=7.5, ha='left', color='#444', fontweight='bold',
        zorder=9,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor='#444', alpha=0.90))


    ax.set_xlim(-0.03, 1.08)
    ax.set_ylim(_Y_MIN, _Y_MAX)
    ax.tick_params(labelsize=8)
    for sp in ax.spines.values(): sp.set_linewidth(0.6)
    ax.set_facecolor('white')
    if ylabel:
        ax.set_ylabel('$C_{LMC}$(TSA)')
    ax.set_xlabel('$H_{norm}$(TSA)')

fig, (ax1, ax2) = plt.subplots(1,2,figsize=(14,7),dpi=300,
                                gridspec_kw={'wspace':0.10})

# ════════════════════════════════════════════════════════════════
# PANEL (a) — Ascending: light scatter + smoothed line on top
# ════════════════════════════════════════════════════════════════
_setup_ax(ax1, ylabel=True)
ax1.set_title('Ascending phase (2010\u20132014)', fontsize=10)

# Light background scatter (honest: shows data density)
ax1.scatter(_df_asc['H_norm_TSA'].values, _df_asc['C_LMC_TSA'].values,
            color=_COL_ASC, marker='o', s=14, alpha=0.16,
            ec='none', zorder=3)

# Smoothed line ABOVE scatter
ax1.plot(_sa['H_norm_TSA'].values, _sa['C_LMC_TSA'].values,
         '-', color=_COL_ASC, lw=1.8, alpha=0.50,
         solid_capstyle='round', solid_joinstyle='round', zorder=4)

# Directional arrow on smooth line
_mid = len(_sa)//2
ax1.annotate('',
    xy=(_sa['H_norm_TSA'].iloc[_mid+1], _sa['C_LMC_TSA'].iloc[_mid+1]),
    xytext=(_sa['H_norm_TSA'].iloc[_mid], _sa['C_LMC_TSA'].iloc[_mid]),
    arrowprops=dict(arrowstyle='->', color=_COL_ASC,
                    lw=2.0, mutation_scale=18), zorder=6)

# Year labels — alternated dx to avoid overlap
_yr_list = ['2010-01','2011-01','2012-01','2013-01','2014-01']
_alt_dx  = [+0.022, -0.085, -0.085, -0.085, -0.085]  # 2010 right, rest left
_alt_ha  = ['left', 'right', 'right', 'right', 'right']
_alt_dy  = [+0.008, +0.008, +0.008, +0.008, +0.008]

for _ym, _dx, _ha, _dy in zip(_yr_list, _alt_dx, _alt_ha, _alt_dy):
    try:
        _t  = pd.Timestamp(_ym)
        _nr = _df_asc.index[np.argmin(np.abs(_df_asc.index - _t))]
        _sr = _sa.loc[_nr]
        ax1.scatter(_sr['H_norm_TSA'], _sr['C_LMC_TSA'],
                    marker='o', s=65, color=_COL_ASC,
                    ec='white', lw=1.0, zorder=7)
        ax1.text(_sr['H_norm_TSA'] + _dx,
                 _sr['C_LMC_TSA'] + _dy,
                 _ym[:4], fontsize=7.5, color=_COL_ASC,
                 ha=_ha, va='bottom', zorder=8)
    except Exception: pass

# Cycle start — open circle ABOVE line
ax1.scatter(df.iloc[0]['H_norm_TSA'], df.iloc[0]['C_LMC_TSA'],
            marker='o', s=90, fc='none', ec=_COL_ASC,
            lw=1.8, zorder=8)
ax1.text(df.iloc[0]['H_norm_TSA'] + 0.022,
         df.iloc[0]['C_LMC_TSA'] - 0.014,
         'Start\n(Jan 2010)', fontsize=7.5,
         color=_COL_ASC, va='top', zorder=8)

# H peak — above everything
ax1.scatter(df.loc[_h_peak_date,'H_norm_TSA'],
            df.loc[_h_peak_date,'C_LMC_TSA'],
            marker='D', s=85, fc='#9C27B0',
            ec='white', lw=0.8, zorder=9)
ax1.text(df.loc[_h_peak_date,'H_norm_TSA'] - 0.022,
         df.loc[_h_peak_date,'C_LMC_TSA'] - 0.015,
         f'H peak\n({_h_peak_date.strftime("%b %Y")})',
         fontsize=7.5, color='#9C27B0',
         ha='right', va='top', zorder=9)

# Panel label — shifted UP (y=0.96 instead of 0.02)
ax1.text(0.015, 0.955, '(a)', transform=ax1.transAxes,
         fontsize=12, fontweight='bold', va='top', zorder=10)

# ════════════════════════════════════════════════════════════════
# PANEL (b) — Descending: original scatter + line
# ════════════════════════════════════════════════════════════════
_setup_ax(ax2, ylabel=False)
ax2.set_title('Descending phase (2014\u20132018)', fontsize=10)

ax2.plot(_df_desc['H_norm_TSA'].values, _df_desc['C_LMC_TSA'].values,
         '-', color=_COL_DESC, lw=1.8, alpha=0.50, zorder=3)
ax2.scatter(_df_desc['H_norm_TSA'].values, _df_desc['C_LMC_TSA'].values,
            color=_COL_DESC, marker='^', s=30, ec='none',
            alpha=0.90, zorder=4)

# Directional arrow
_mid2 = len(_df_desc)//2
ax2.annotate('',
    xy=(_df_desc['H_norm_TSA'].iloc[_mid2+1],
        _df_desc['C_LMC_TSA'].iloc[_mid2+1]),
    xytext=(_df_desc['H_norm_TSA'].iloc[_mid2],
            _df_desc['C_LMC_TSA'].iloc[_mid2]),
    arrowprops=dict(arrowstyle='->', color=_COL_DESC,
                    lw=2.0, mutation_scale=18), zorder=6)

# Year labels descending — alternated to avoid overlap with curve
_yr_list_d  = ['2015-01','2016-01','2017-01','2018-01']
_alt_dx_d   = [+0.022, +0.022, -0.085, -0.085]
_alt_ha_d   = ['left',  'left', 'right', 'right']
for _ym, _dx, _ha in zip(_yr_list_d, _alt_dx_d, _alt_ha_d):
    try:
        _t  = pd.Timestamp(_ym)
        _nr = _df_desc.index[np.argmin(np.abs(_df_desc.index - _t))]
        ax2.scatter(df.loc[_nr,'H_norm_TSA'], df.loc[_nr,'C_LMC_TSA'],
                    marker='^', s=60, color=_COL_DESC,
                    ec='white', lw=1.0, zorder=7)
        ax2.text(df.loc[_nr,'H_norm_TSA'] + _dx,
                 df.loc[_nr,'C_LMC_TSA'] + 0.007,
                 _ym[:4], fontsize=7.5, color=_COL_DESC,
                 ha=_ha, va='bottom', zorder=8)
    except Exception: pass

# C_LMC peak — top priority z-order
ax2.scatter(df.loc[_c_peak_date,'H_norm_TSA'],
            df.loc[_c_peak_date,'C_LMC_TSA'],
            marker='*', s=220, color='#FF6F00',
            ec='white', lw=0.8, zorder=9)
ax2.text(df.loc[_c_peak_date,'H_norm_TSA'] + 0.022,
         df.loc[_c_peak_date,'C_LMC_TSA'] + 0.007,
         f'$C_{{LMC}}$ peak\n({_c_peak_date.strftime("%b %Y")})',
         fontsize=7.5, color='#FF6F00', va='bottom', zorder=9)

# H peak reference
ax2.scatter(df.loc[_h_peak_date,'H_norm_TSA'],
            df.loc[_h_peak_date,'C_LMC_TSA'],
            marker='D', s=85, fc='#9C27B0',
            ec='white', lw=0.8, zorder=9)
ax2.text(df.loc[_h_peak_date,'H_norm_TSA'] - 0.022,
         df.loc[_h_peak_date,'C_LMC_TSA'] - 0.015,
         f'H peak\n({_h_peak_date.strftime("%b %Y")})',
         fontsize=7.5, color='#9C27B0',
         ha='right', va='top', zorder=9)

# Cycle end
ax2.scatter(df.iloc[-1]['H_norm_TSA'], df.iloc[-1]['C_LMC_TSA'],
            marker='o', s=90, fc='none', ec=_COL_DESC,
            lw=1.8, zorder=8)
ax2.text(df.iloc[-1]['H_norm_TSA'] + 0.022,
         df.iloc[-1]['C_LMC_TSA'] - 0.014,
         'End\n(Oct 2018)', fontsize=7.5,
         color=_COL_DESC, va='top', zorder=8)

# Panel label — shifted UP
ax2.text(0.015, 0.955, '(b)', transform=ax2.transAxes,
         fontsize=12, fontweight='bold', va='top', zorder=10)

# ── Shared legend ──────────────────────────────────────────────────────────────
_leg = [
    mlines.Line2D([],[],color='#AAAAAA',lw=2,
                  label='$C_{max}$'),
    mlines.Line2D([],[],color=_COL_ASC, lw=3.0,
                  label='Ascending'),
    mlines.Line2D([],[],color=_COL_DESC,lw=1.8,
                  marker='^',ms=6,
                  label='Descending'),
]
fig.legend(handles=_leg, loc='upper center', ncol=3,
           fontsize=9, bbox_to_anchor=(0.5, 1.015),
           prop={'family': MONO, 'size': 9})

fig.patch.set_facecolor('white')
plt.savefig('8.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Verification
print('8.png saved')
print(f'Y-axis range: [{_Y_MIN:.3f}, {_Y_MAX:.3f}]')
print(f'C_LMC data range: [{df["C_LMC_TSA"].min():.4f}, {df["C_LMC_TSA"].max():.4f}]')
print(f'H_norm data range: [{df["H_norm_TSA"].min():.3f}, {df["H_norm_TSA"].max():.3f}]')
print(f'H peak: {_h_peak_date.strftime("%Y-%m")}  '
      f'H_norm={df.loc[_h_peak_date,"H_norm_TSA"]:.3f}')
print(f'C peak: {_c_peak_date.strftime("%Y-%m")}  '
      f'C_LMC={df.loc[_c_peak_date,"C_LMC_TSA"]:.4f}')

In [ ]:
# CELL_3D_SUPP — Fig. S1 (supplementary): C-H plane with time as depth axis
# X = H_norm, Y = fractional year (time), Z = C_LMC
# View angle chosen so both branches are visible and separated temporally
# Output: 9_CH_3D_supp.png | DPI 300

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.font_manager as fm
import matplotlib.dates as mdates
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

_avail = {f.name for f in fm.fontManager.ttflist}
MONO = next((f for f in ['Courier Prime','Courier New','DejaVu Sans Mono']
             if f in _avail), 'monospace')
plt.rcParams.update({'font.family':'monospace','font.monospace':[MONO],'font.size':9})

df = df_dynamic.copy().sort_index()
_h_peak_date = df['H_TSA'].idxmax()
_c_peak_date  = df['C_LMC_TSA'].idxmax()
_df_asc  = df[df.index <= _h_peak_date]
_df_desc = pd.concat([df.loc[[_h_peak_date]], df[df.index > _h_peak_date]])
_COL_ASC, _COL_DESC = '#2171B5', '#D84315'

# Fractional year
def _frac_year(ts): return ts.year + (ts.dayofyear - 1) / 365.25
df['frac_year'] = [_frac_year(t) for t in df.index]
_df_asc['frac_year']  = [_frac_year(t) for t in _df_asc.index]
_df_desc['frac_year'] = [_frac_year(t) for t in _df_desc.index]

# Theoretical C_max (projected to the H-C back wall at Y=max_year)
try:
    _H_th,_C_th = lmc_upper_bound(N_bins=10,n_points=8000)
    _s=np.argsort(_H_th); _H_th,_C_th=_H_th[_s],_C_th[_s]
    _hb=np.linspace(0,1,600); _cm=np.full_like(_hb,np.nan)
    for _k,_h in enumerate(_hb):
        _m=np.abs(_H_th-_h)<0.012
        if _m.any(): _cm[_k]=_C_th[_m].max()
    _v=~np.isnan(_cm); _cs=np.full_like(_cm,np.nan)
    _cs[_v]=gaussian_filter1d(_cm[_v].astype(float),sigma=2.5)
    _has_c=True
except NameError: _has_c=False

fig = plt.figure(figsize=(12, 9), dpi=300)
ax  = fig.add_subplot(111, projection='3d')

_y_min = df['frac_year'].min()
_y_max = df['frac_year'].max()

# C_max projected on back wall (Y = y_max)
if _has_c:
    _valid_h = _hb[~np.isnan(_cs)]
    _valid_c = _cs[~np.isnan(_cs)]
    ax.plot(_valid_h, np.full_like(_valid_h, _y_max), _valid_c,
            '-', color='#AAAAAA', lw=1.8, alpha=0.7, zorder=1,
            label='$C_{max}$ projection')

# White noise projection line (H≈1, C≈0, all years)
ax.plot([1.0, 1.0], [_y_min, _y_max], [0.0, 0.0],
        '--', color='#888888', lw=1.0, alpha=0.6)

# Ascending branch — blue
ax.plot(_df_asc['H_norm_TSA'].values,
        _df_asc['frac_year'].values,
        _df_asc['C_LMC_TSA'].values,
        '-', color=_COL_ASC, lw=2.5, alpha=0.90, zorder=3,
        label='Ascending (2010\u20132014)')
ax.scatter(_df_asc['H_norm_TSA'].values,
           _df_asc['frac_year'].values,
           _df_asc['C_LMC_TSA'].values,
           color=_COL_ASC, marker='o', s=18, alpha=0.65, zorder=4,
           depthshade=True)

# Descending branch — red
ax.plot(_df_desc['H_norm_TSA'].values,
        _df_desc['frac_year'].values,
        _df_desc['C_LMC_TSA'].values,
        '-', color=_COL_DESC, lw=2.5, alpha=0.90, zorder=3,
        label='Descending (2014\u20132018)')
ax.scatter(_df_desc['H_norm_TSA'].values,
           _df_desc['frac_year'].values,
           _df_desc['C_LMC_TSA'].values,
           color=_COL_DESC, marker='^', s=20, alpha=0.65, zorder=4,
           depthshade=True)

# Key event markers
def _mark3(ax, row, mk, col, sz, label=None):
    ax.scatter([row['H_norm_TSA']], [row['frac_year']], [row['C_LMC_TSA']],
               marker=mk, s=sz, color=col, edgecolors='white', lw=0.8,
               zorder=6, label=label, depthshade=False)

_mark3(ax, df.iloc[0],                       'o',  _COL_ASC, 120)
_mark3(ax, df.loc[_h_peak_date],              'D',  '#9C27B0',110)
_mark3(ax, df.loc[_c_peak_date],              '*',  '#FF6F00',180)
_mark3(ax, df.iloc[-1],                       'o',  _COL_DESC, 90)

# Annotations
ax.text(df.iloc[0]['H_norm_TSA']+0.04,
        df.iloc[0]['frac_year']-0.15,
        df.iloc[0]['C_LMC_TSA']+0.005,
        'Start\nJan 2010', fontsize=7.5, color=_COL_ASC)
ax.text(df.loc[_h_peak_date,'H_norm_TSA']+0.04,
        _frac_year(_h_peak_date)+0.1,
        df.loc[_h_peak_date,'C_LMC_TSA']+0.005,
        f'H peak\n{_h_peak_date.strftime("%b %Y")}',
        fontsize=7.5, color='#9C27B0')
ax.text(df.loc[_c_peak_date,'H_norm_TSA']+0.04,
        _frac_year(_c_peak_date)+0.1,
        df.loc[_c_peak_date,'C_LMC_TSA']+0.008,
        f'$C_{{LMC}}$ peak\n{_c_peak_date.strftime("%b %Y")}',
        fontsize=7.5, color='#FF6F00')
ax.text(df.iloc[-1]['H_norm_TSA']+0.04,
        _frac_year(df.index[-1])+0.1,
        df.iloc[-1]['C_LMC_TSA']+0.005,
        'End Oct 2018', fontsize=7.5, color=_COL_DESC)

# Shadow projections on the floor (Y=const floor at C_LMC=0)
ax.plot(_df_asc['H_norm_TSA'].values,
        _df_asc['frac_year'].values,
        np.zeros(len(_df_asc)),
        '-', color=_COL_ASC, lw=0.8, alpha=0.20, zorder=1)
ax.plot(_df_desc['H_norm_TSA'].values,
        _df_desc['frac_year'].values,
        np.zeros(len(_df_desc)),
        '-', color=_COL_DESC, lw=0.8, alpha=0.20, zorder=1)

# Axes setup
ax.set_xlabel('$H_{norm}$(TSA)', labelpad=10)
ax.set_ylabel('Time', labelpad=10)
ax.set_zlabel('$C_{LMC}$(TSA)', labelpad=10)
ax.set_xlim(0.0, 1.05)
ax.set_ylim(_y_min - 0.2, _y_max + 0.2)
ax.set_zlim(0.0, 0.32)
ax.set_yticks([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018])
ax.set_yticklabels(['2011','2012','2013','2014','2015','2016','2017','2018'],
                   fontsize=7.5)
ax.tick_params(labelsize=7.5)

# View angle: time axis as depth, separating both branches
ax.view_init(elev=22, azim=-55)

ax.legend(fontsize=8.5, loc='upper left',
          prop={'family': MONO, 'size': 8.5})
ax.set_title('3D temporal embedding of the SC24 $C$–$H$ trajectory\n'
             '(Supplementary material)', fontsize=10, pad=12)

fig.patch.set_facecolor('white')
ax.set_facecolor('white')
plt.tight_layout()
plt.savefig('9_CH_3D_supp.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('9_CH_3D_supp.png saved')
print(f'H_peak: {_h_peak_date.strftime("%Y-%m")}  '
      f'H_norm={df.loc[_h_peak_date,"H_norm_TSA"]:.3f}  '
      f'C_LMC={df.loc[_h_peak_date,"C_LMC_TSA"]:.4f}')
print(f'C_peak: {_c_peak_date.strftime("%Y-%m")}  '
      f'C_LMC={df.loc[_c_peak_date,"C_LMC_TSA"]:.4f}')

In [ ]:
# CELL FIG10A — Fig. 10 variante A: I(t) + C_LMC(t) em dois painéis alinhados
# Histerese visível pela assimetria temporal (I e C_LMC fora de fase)
# Output: 10A.png

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd

_avail = {f.name for f in fm.fontManager.ttflist}
MONO = next((f for f in ['Courier Prime','Courier New','DejaVu Sans Mono']
             if f in _avail), 'monospace')
plt.rcParams.update({'font.family':'monospace','font.monospace':[MONO],'font.size':9})

df = df_dynamic.copy()
_mi_peak = df['MI'].idxmax()
_c_peak  = df['C_LMC_TSA'].idxmax()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.2, 6.0), dpi=300,
                                sharex=True, gridspec_kw={'hspace': 0.10})

# Phase shading (both panels)
for ax in (ax1, ax2):
    ax.axvspan(df.index.min(), _mi_peak,   alpha=0.10, color='#2171B5', zorder=0)
    ax.axvspan(_mi_peak, _c_peak,          alpha=0.10, color='#FF6F00', zorder=0)
    ax.axvspan(_c_peak, df.index.max(),    alpha=0.10, color='#D84315', zorder=0)
    ax.axvline(_mi_peak, color='#2171B5', lw=0.8, ls='--', alpha=0.6)
    ax.axvline(_c_peak,  color='#FF6F00', lw=0.8, ls='--', alpha=0.6)

# Panel 1: I(TSA;PSI) vs time
ax1.plot(df.index, df['MI'], color='#2171B5', lw=2.0, zorder=3)
ax1.fill_between(df.index, df['MI_lo'], df['MI_hi'],
                 color='#2171B5', alpha=0.20, zorder=2)
ax1.set_ylabel('I(TSA; PSI) (bits)', fontsize=9)
ax1.tick_params(labelsize=8)
ax1.text(0.02, 0.90, 'I(TSA;PSI)', transform=ax1.transAxes,
         fontsize=9, color='#2171B5', fontweight='bold')

# Annotate MI peak
ax1.annotate(f'MI peak\n{_mi_peak.strftime("%b %Y")}',
             xy=(_mi_peak, df.loc[_mi_peak,'MI']),
             xytext=(_mi_peak - pd.DateOffset(months=18),
                     df['MI'].max()*0.88),
             fontsize=7.5, color='#2171B5',
             arrowprops=dict(arrowstyle='->', color='#2171B5', lw=0.9))

# Panel 2: C_LMC(TSA) vs time
ax2.plot(df.index, df['C_LMC_TSA'], color='#E6550D', lw=2.0, zorder=3)
ax2.plot(df.index, df['C_LMC_PSI'], color='#E6550D', lw=1.2,
         ls='--', alpha=0.55, zorder=3, label='$C_{LMC}$(PSI)')
ax2.set_ylabel('$C_{LMC}$(TSA)', fontsize=9)
ax2.set_xlabel('Year', fontsize=9)
ax2.tick_params(labelsize=8)
ax2.text(0.02, 0.90, '$C_{LMC}$(TSA)', transform=ax2.transAxes,
         fontsize=9, color='#E6550D', fontweight='bold')

# Annotate C_LMC peak
ax2.annotate(f'$C_{{LMC}}$ peak\n{_c_peak.strftime("%b %Y")}',
             xy=(_c_peak, df.loc[_c_peak,'C_LMC_TSA']),
             xytext=(_c_peak + pd.DateOffset(months=8),
                     df['C_LMC_TSA'].max()*0.88),
             fontsize=7.5, color='#E6550D',
             arrowprops=dict(arrowstyle='->', color='#E6550D', lw=0.9))

# Phase labels (top panel)
_phases = [
    (df.index.min(), _mi_peak, 'Ascending\n2010–2014', '#2171B5'),
    (_mi_peak, _c_peak,        'Maximum\n2014–2016',   '#FF6F00'),
    (_c_peak, df.index.max(),  'Descending\n2016–2018','#D84315'),
]
for _t0,_t1,_lbl,_col in _phases:
    _tm = _t0 + (_t1 - _t0)/2
    ax1.text(_tm, ax1.get_ylim()[1]*0.98 if ax1.get_ylim()[1]>0 else 2.4,
             _lbl, ha='center', va='top', fontsize=7.5,
             color=_col, fontweight='bold')

# Asimetry note
ax2.text(0.98, 0.06,
         'Hysteresis: I peaks before $C_{LMC}$\n(temporal offset = phase asymmetry)',
         transform=ax2.transAxes, fontsize=7.5, ha='right',
         bbox=dict(facecolor='white', edgecolor='#CCC', alpha=0.90,
                   boxstyle='round,pad=0.35'))

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.xaxis.set_major_locator(mdates.YearLocator(2))

for ax in (ax1, ax2):
    ax.set_xlim(df.index.min(), df.index.max())
    for sp in ax.spines.values(): sp.set_linewidth(0.6)
    ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.savefig('9.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show(); print('9.png saved  [was 10A.png — renumbered after Fig.8 elimination]')

### ~~CELL F9.5 — Fig. 10 old (I×C_LMC phase space)~~

**DEPRECATED** — Replaced by **CELL FIG10A** which shows I(t) + C_LMC(t) temporal panels (`9.png`).

Original code preserved as comment below for reproducibility.


## Section 10 — Reference Table and Final Gate Check

In [ ]:
# CELL 18 — tabela_referencia_v2.csv (single source of truth for manuscript numbers)
# F8b: C_LMC entries appended here — Cell 18 is the sole owner of this CSV.
ref = []

# ── Tabela II estática ─────────────────────────────────────────────────────────
for scale, row in df_table_II.iterrows():
    for metric in ['H(TSA)','H(PSI)','PE(TSA)','PE(PSI)','LZC(TSA)','LZC(PSI)','I(TSA;PSI)']:
        ref.append({'scale':scale,'metric':metric,
                    'value':row[metric],'ic_lo_95':np.nan,'ic_hi_95':np.nan})

# ── Extremos dinâmicos (janela 26 meses) ──────────────────────────────────────
for col in ['H_TSA','H_PSI','LZC_TSA','LZC_PSI','MI','PE_TSA','PE_PSI']:
    lc = col+'_lo' if col+'_lo' in df_dynamic.columns else None
    hc = col+'_hi' if col+'_hi' in df_dynamic.columns else None
    for sfx, fn in [('max', df_dynamic[col].idxmax),
                    ('min', df_dynamic[col].idxmin)]:
        ix = fn()
        ref.append({'scale': f'26m_window_{sfx}', 'metric': col,
                    'value':     df_dynamic.loc[ix, col],
                    'ic_lo_95': df_dynamic.loc[ix, lc] if lc else np.nan,
                    'ic_hi_95': df_dynamic.loc[ix, hc] if hc else np.nan})

# ── Estatísticas de fase (Pearson, KS) ───────────────────────────────────────
ref += [
    {'scale':'fig7','metric':'Pearson_r_H_LZC','value':r_tsa,    'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'fig7','metric':'Pearson_p_H_LZC','value':p_tsa,    'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'fig6','metric':'KS_D_MI',         'value':ks_D_MI,  'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'fig6','metric':'KS_p_MI',         'value':ks_p_MI,  'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'fig6','metric':'KS_D_LZC_TSA',    'value':ks_D_LZC, 'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'fig6','metric':'KS_p_LZC_TSA',    'value':ks_p_LZC, 'ic_lo_95':np.nan,'ic_hi_95':np.nan},
]

# ── C_LMC (F8b) ───────────────────────────────────────────────────────────────
idx_cmax_tsa = df_dynamic['C_LMC_TSA'].idxmax()
idx_cmin_tsa = df_dynamic['C_LMC_TSA'].idxmin()
idx_cmax_psi = df_dynamic['C_LMC_PSI'].idxmax()
idx_cmin_psi = df_dynamic['C_LMC_PSI'].idxmin()

ref += [
    {'scale':'monthly_window','metric':'C_LMC_TSA_max',
     'value': df_dynamic['C_LMC_TSA'].max(),         'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'monthly_window','metric':'C_LMC_TSA_min',
     'value': df_dynamic['C_LMC_TSA'].min(),         'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'monthly_window','metric':'C_LMC_PSI_max',
     'value': df_dynamic['C_LMC_PSI'].max(),         'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'monthly_window','metric':'C_LMC_PSI_min',
     'value': df_dynamic['C_LMC_PSI'].min(),         'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'monthly_window','metric':'H_norm_TSA_at_C_max',
     'value': df_dynamic.loc[idx_cmax_tsa,'H_norm_TSA'], 'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'monthly_window','metric':'H_norm_TSA_at_C_min',
     'value': df_dynamic.loc[idx_cmin_tsa,'H_norm_TSA'], 'ic_lo_95':np.nan,'ic_hi_95':np.nan},
    {'scale':'monthly_window','metric':'C_LMC_TSA_max_date',
     'value': float(idx_cmax_tsa.year) + float(idx_cmax_tsa.month-1)/12,
     'ic_lo_95':np.nan,'ic_hi_95':np.nan},
]

df_ref = pd.DataFrame(ref)
df_ref.to_csv('tabela_referencia_v2.csv', index=False)
print(f'tabela_referencia_v2.csv: {len(df_ref)} entradas (41 originais + 7 C_LMC)')
assert len(df_ref) == 48, f"Esperado 48, encontrado {len(df_ref)}"

In [ ]:
# CELL 19 — Gate checks + relatório automático
# Roda todos os asserts de integridade, imprime resumo estruturado
# e salva RELATORIO_NOTEBOOK_LOCAL.md no diretório de trabalho.
# UPDATED: filenames updated for new figure scheme (F9-final)

import platform, datetime, scipy

PASS, FAIL = '[ OK ]', '[FAIL]'
erros = []

def chk(cond, label, detail=''):
    if not cond:
        erros.append(f'{label}: {detail}')
    return PASS if cond else FAIL

linhas = []

def sec(titulo):
    linhas.extend(['', '─' * 60, titulo, '─' * 60])

def row(status, label, val='', esperado=''):
    esp = f'  (esperado: {esperado})' if esperado else ''
    linhas.append(f'{status}  {label:<42s}{val}{esp}')

# ── CABEÇALHO ─────────────────────────────────────────────────────────────────
linhas += [
    '=' * 60,
    'RELATORIO_NOTEBOOK_LOCAL — SC24_info_analysis.ipynb',
    'Gerado automaticamente pelo Cell 19',
    f'Data/hora : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    f'Python    : {platform.python_version()}',
    f'pandas {pd.__version__}  |  numpy {np.__version__}  |  scipy {scipy.__version__}',
    '=' * 60,
]

# ── BLOCO 0 — Integridade dos dados ───────────────────────────────────────────
sec('BLOCO 0 — Integridade do catálogo (Cell 2)')
n_obs = len(df_sc24)
d_max = df_sc24.index.max()
d_min = df_sc24.index.min()
row(chk(n_obs == 3636,  'N obs'),  'N observações',   str(n_obs),        '3636')
row(chk(d_max == pd.Timestamp('2019-10-02'), 'data max'),
    'Último registro',  str(d_max.date()), '2019-10-02')
row(chk(d_min == pd.Timestamp('2008-12-01'), 'data min'),
    'Primeiro registro', str(d_min.date()), '2008-12-01')

# ── BLOCO 1 — Tabela II ────────────────────────────────────────────────────────
sec('BLOCO 1 — Tabela II (Cell 6)')
REF_II = {
    ('Daily',  'H(TSA)'    ): 1.7789, ('Daily',  'H(PSI)'    ): 1.3091,
    ('Daily',  'PE(TSA)'   ): 0.9045, ('Daily',  'PE(PSI)'   ): 0.8231,  # [FIX F-PE]
    ('Daily',  'LZC(TSA)'  ): 0.1254, ('Daily',  'LZC(PSI)'  ): 0.0987,
    ('Daily',  'I(TSA;PSI)'): 0.7557,
    ('Monthly','H(TSA)'    ): 2.7615, ('Monthly','H(PSI)'    ): 2.6063,
    ('Monthly','PE(TSA)'   ): 0.9973, ('Monthly','PE(PSI)'   ): 0.9838,  # [FIX F-PE]
    ('Monthly','LZC(TSA)'  ): 0.3893, ('Monthly','LZC(PSI)'  ): 0.3664,
    ('Monthly','I(TSA;PSI)'): 2.0433,
    ('Annual', 'H(TSA)'    ): 2.6258, ('Annual', 'H(PSI)'    ): 2.5850,
    ('Annual', 'PE(TSA)'   ): 0.6520, ('Annual', 'PE(PSI)'   ): 0.6520,
    ('Annual', 'LZC(TSA)'  ): 0.7500, ('Annual', 'LZC(PSI)'  ): 0.7500,
    ('Annual', 'I(TSA;PSI)'): 2.3554,
}
for (scale, met), ref in REF_II.items():
    val = df_table_II.loc[scale, met]
    ok  = abs(val - ref) < 0.002
    if not ok: erros.append(f'Tabela II {scale} {met}: {val:.4f} (ref {ref})')
    row(PASS if ok else FAIL, f'{scale:8s} {met}', f'{val:.4f}', f'{ref}')

h_d = df_table_II.loc['Daily', 'H(TSA)']
row(chk(abs(h_d - np.log2(10)) > 0.05, 'qcut guard'),
    'Guard qcut  H(TSA) daily ≠ log2(10)', f'{h_d:.6f}', f'≠ {np.log2(10):.4f}')

# ── BLOCO 2 — Estabilidade de bins ────────────────────────────────────────────
sec('BLOCO 2 — Estabilidade de bins (Cell 7)')
row(chk(N_BINS == 10, 'N_BINS'), 'N_BINS', str(N_BINS), '10')
try:
    mi_10 = df_stab_monthly.loc[10, 'MI']
    row(PASS, 'MI(n=10) monthly', f'{mi_10:.4f}')
except Exception as e:
    row(FAIL, 'MI(n=10) monthly', str(e))

# ── BLOCO 3 — Janela ótima ────────────────────────────────────────────────────
sec('BLOCO 3 — Janela ótima (Cell 9)')
row(chk(WINDOW == 26, 'WINDOW'), 'WINDOW (meses)', str(WINDOW), '26')
row(PASS, 'KneeLocator elbow', str(cotovelo))
row(PASS, 'Nota: curva sem elbow clássico após qcut; 26m via QBO', '')

# ── BLOCO 4 — Colunas df_dynamic ──────────────────────────────────────────────
sec('BLOCO 4 — Colunas df_dynamic (Cell 11)')
n_win = len(df_dynamic.dropna(subset=['MI']))
row(chk(n_win == 106, 'N janelas'), 'N janelas de 26 meses', str(n_win), '106')
for col in ['MI_lo','MI_hi','H_TSA_lo','H_TSA_hi','H_PSI_lo','H_PSI_hi',
            'LZC_TSA_lo','LZC_TSA_hi','PE_TSA','PE_PSI']:
    ok = col in df_dynamic.columns
    if not ok: erros.append(f'Coluna ausente: {col}')
    row(PASS if ok else FAIL, f'Coluna {col}')

# ── BLOCO 5 — Mann-Whitney U ──────────────────────────────────────────────────
sec('BLOCO 5 — Mann-Whitney U (Cell 13)')
# [FIX F-PE] PE exempt from phase-discrimination gate: lack of min/max
# separation (p~0.76 for TSA) is a reported result, not a failure.
_PE_EXEMPT = {'PE_TSA', 'PE_PSI'}
for met, vals in comparison.items():
    mn_min = vals['Min_mean']
    mn_max = vals['Max_mean']
    pv     = vals['p_value']
    if met in _PE_EXEMPT:
        row(PASS, f'{met:<12s} min={mn_min:.4f}  max={mn_max:.4f}',
            f'p={pv:.2e}', 'PE: no phase sep. (expected)')
        continue
    ok = pv < 2e-10
    if not ok: erros.append(f'MW {met}: p={pv:.2e} não < 2e-10')
    row(PASS if ok else FAIL,
        f'{met:<12s} min={mn_min:.4f}  max={mn_max:.4f}',
        f'p={pv:.2e}', 'p < 2e-10')

# ── BLOCO 6 — KS test ─────────────────────────────────────────────────────────
sec('BLOCO 6 — KS test fases (CELL KS)')
row(chk(abs(ks_D_MI  - 0.3962) < 0.005, 'KS_D_MI'),
    'KS(MI)  D',    f'{ks_D_MI:.4f}',  '0.3962 ±0.005')
row(chk(abs(ks_p_MI  - 0.0004) < 0.001, 'KS_p_MI'),
    'KS(MI)  p',    f'{ks_p_MI:.4f}',  '0.0004 ±0.001')
row(chk(abs(ks_D_LZC - 0.3585) < 0.005, 'KS_D_LZC'),
    'KS(LZC) D',    f'{ks_D_LZC:.4f}', '0.3585 ±0.005')
row(chk(abs(ks_p_LZC - 0.0020) < 0.002, 'KS_p_LZC'),
    'KS(LZC) p',    f'{ks_p_LZC:.4f}', '0.0020 ±0.002')
row(PASS, 'SC24_MAX (split)',    '2014-06-01', '53/53 simétrico')
row(PASS, 'Direção do loop',     loop_dir,     'CW')
row(PASS, 'Gap de fechamento',   f'{gap_pct}%', '~10%')

# ── BLOCO 7 — Pearson r ───────────────────────────────────────────────────────
sec('BLOCO 7 — Pearson r (Cell 17)')
row(chk(abs(r_tsa - 0.9656) < 0.002, 'Pearson r'),
    'r(H_TSA, LZC_TSA)', f'{r_tsa:.6f}', '0.9656 ±0.002')
row(PASS, 'p-valor', f'{p_tsa:.4e}', '~1.14e-62')

# ── BLOCO 8 — tabela_referencia_v2.csv ────────────────────────────────────────
sec('BLOCO 8 — tabela_referencia_v2.csv (Cell 18)')
n_ref = len(pd.read_csv('tabela_referencia_v2.csv'))
row(chk(n_ref == 48, 'n_ref'), 'N entradas na tabela de referência', str(n_ref), '48')

# ── BLOCO 9 — Arquivos de saída ───────────────────────────────────────────────
sec('BLOCO 9 — Arquivos de saída')
# Canonical filenames after F9-final figure reorganisation:
#   Fig 1-2, 5, 7: old names (renamed to N.png by CELL_F11_RENAME)
#   Fig 3, 4, 6  : already N.png
#   Fig 8 (C-H)  : 9_CH.png (renamed to 8.png by F11_RENAME)
#   Fig 9 (temporal): 9.png
figs_esperados = [
    '1.png',
    '2.png',
    '3.png',
    '4.png',
    '5.png',
    '6.png',
    '7.png',
    'tabela_referencia_v2.csv',
    'df_resultados_janela_v2.csv',
]
for fn in figs_esperados:
    ok = os.path.exists(fn)
    if not ok: erros.append(f'Arquivo ausente: {fn}')
    row(PASS if ok else FAIL, fn)

# ── BLOCO F9 — Figuras geradas (F9-final) ─────────────────────────────────────
import os as _os

sec('BLOCO F9 — Figuras geradas (F9-final)')

# Manuscript figure → generated filename mapping
_f9_map = {
    '3.png':    'Fig. 3  window stability (QBO + sigma)',
    '4.png':    'Fig. 4  dynamic metrics + phase shading',
    '6.png':    'Fig. 6  D_JS(t) + I(t) temporal panels',
    '8.png': 'Fig. 8  C-H plane (→ 8.png via F11_RENAME)',
    '9.png':    'Fig. 9  I(t) + C_LMC(t) temporal hysteresis',
}
for _fn, _desc in _f9_map.items():
    _ok = _os.path.exists(_fn)
    if not _ok: erros.append(f'Arquivo ausente: {_fn}')
    row(chk(_ok, _fn), f'{_fn}  {_desc}',
        'EXISTS' if _ok else 'NOT FOUND', 'EXISTS')

# C_LMC peak value
_idx_cmax_f9 = df_dynamic['C_LMC_TSA'].idxmax()
_val_cmax_f9 = df_dynamic.loc[_idx_cmax_f9, 'C_LMC_TSA']
row(chk(abs(_val_cmax_f9 - 0.2371) < 0.001, 'C_LMC_TSA max'),
    'C_LMC_TSA máximo (Fig. 9 lower panel)',
    f'{_val_cmax_f9:.4f}  ({_idx_cmax_f9.strftime("%Y-%m-%d")})',
    'calculado')

# Loop direction (I × C_LMC plane)
_xl = df_dynamic['MI'].values
_yl = df_dynamic['C_LMC_TSA'].values
_sa = 0.5 * np.sum(_xl[:-1] * _yl[1:] - _xl[1:] * _yl[:-1])
_loop10 = 'CW' if _sa < 0 else 'CCW'
row(chk(_loop10 == 'CW', 'loop_ICLMC_dir'),
    'Loop I×C_LMC direction', _loop10, 'CW (expected)')

# ── BLOCO 10 — Extremos dinâmicos ─────────────────────────────────────────────
sec('BLOCO 10 — Extremos da janela de 26 meses')
REF_DYN = {
    'H_TSA' : (2.9210, 0.6219), 'H_PSI' : (2.9801, 0.6219),
    'LZC_TSA': (0.6538, 0.3462),'LZC_PSI': (0.6923, 0.3462),
    'MI'    : (2.3357, 0.4410), 'PE_TSA': (0.9941, 0.8794),  # [FIX F-PE]
    'PE_PSI': (0.9941, 0.8659),  # [FIX F-PE]
}
for col, (ref_max, ref_min) in REF_DYN.items():
    vmax = df_dynamic[col].max()
    vmin = df_dynamic[col].min()
    ok_max = abs(vmax - ref_max) < 0.005
    ok_min = abs(vmin - ref_min) < 0.005
    if not ok_max: erros.append(f'{col} max: {vmax:.4f} (ref {ref_max})')
    if not ok_min: erros.append(f'{col} min: {vmin:.4f} (ref {ref_min})')
    row(PASS if ok_max else FAIL, f'{col} max', f'{vmax:.4f}', f'{ref_max}')
    row(PASS if ok_min else FAIL, f'{col} min', f'{vmin:.4f}', f'{ref_min}')

# ── BLOCO 11 — C_LMC (F8b) ────────────────────────────────────────────────────
sec('BLOCO 11 — C_LMC (F8b)')
for col in ['C_LMC_TSA','C_LMC_PSI','H_norm_TSA','H_norm_PSI','D_JS_TSA','D_JS_PSI']:
    ok = col in df_dynamic.columns
    if not ok: erros.append(f'Coluna ausente: {col}')
    row(PASS if ok else FAIL, f'Coluna {col}', 'presente' if ok else 'AUSENTE', 'presente')

n_cols = df_dynamic.shape[1]
ok_cols = (n_cols == 23)
if not ok_cols: erros.append(f'df_dynamic colunas: {n_cols} (esperado 23)')
row(PASS if ok_cols else FAIL, 'df_dynamic total colunas', str(n_cols), '23')

n_ref_new = len(pd.read_csv('tabela_referencia_v2.csv'))
ok_ref = (n_ref_new == 48)
if not ok_ref: erros.append(f'tabela_referencia linhas: {n_ref_new} (esperado 48)')
row(PASS if ok_ref else FAIL, 'tabela_referencia_v2.csv linhas', str(n_ref_new), '48')

c_max_v = df_dynamic['C_LMC_TSA'].max()
c_min_v = df_dynamic['C_LMC_TSA'].min()
fase_max = df_dynamic['C_LMC_TSA'].idxmax()
row(PASS, 'C_LMC_TSA máximo', f'{c_max_v:.4f}  ({fase_max.date()})', 'calculado')
row(PASS, 'C_LMC_TSA mínimo',
    f'{c_min_v:.4f}  ({df_dynamic["C_LMC_TSA"].idxmin().date()})', 'calculado')
row(PASS, 'C_LMC_PSI máximo', f'{df_dynamic["C_LMC_PSI"].max():.4f}', 'calculado')

ok_curve = ('H_curve' in dir() and len(H_curve) > 0)
if not ok_curve: erros.append('Curvas teóricas lmc_upper_bound ausentes')
row(PASS if ok_curve else FAIL,
    'Curvas teóricas C_max(H_norm)',
    f'{len(H_curve)} pts' if ok_curve else 'AUSENTE', '>0 pts')

h_check = df_dynamic['H_TSA'].mean()
ok_h = abs(h_check - np.log2(10)) > 0.05
if not ok_h: erros.append(f'H_TSA médio = {h_check:.4f} ≈ log2(10) — qcut ativo!')
row(PASS if ok_h else FAIL, 'H_TSA médio ≠ log2(10)',
    f'{h_check:.4f}', f'≠ {np.log2(10):.4f}')

# ── SUMÁRIO FINAL ──────────────────────────────────────────────────────────────
sec('SUMÁRIO FINAL')
if erros:
    row(FAIL, f'{len(erros)} falha(s) encontrada(s)')
    for e in erros:
        linhas.append(f'         !! {e}')
    linhas.extend(['', 'Gate F9-final: REPROVADO — corrigir antes de prosseguir'])
else:
    row(PASS, 'Todos os checks passaram — zero falhas')
    linhas.extend(['', 'Gate F9-final: APROVADO'])
linhas.append('=' * 60)

# ── IMPRIMIR E SALVAR ──────────────────────────────────────────────────────────
relatorio = '\n'.join(linhas)
print(relatorio)

with open('RELATORIO_NOTEBOOK_LOCAL.md', 'w', encoding='utf-8') as f:
    f.write('```\n' + relatorio + '\n```\n')
print('\nRELATORIO_NOTEBOOK_LOCAL.md salvo.')

In [ ]:
# CELL F11_RENAME — Canonical figure filenames for submission package
# Run ONCE after all figures approved. Idempotent.
# Maps all output filenames to canonical N.png scheme.
import os, shutil

_rename_map = {
    # source (generated by notebook) → canonical (for main.tex)
    '1.png'        : '1.png',
    '2.png'     : '2.png',
    '3.png'                       : '3.png',   # already canonical
    '4.png'                       : '4.png',   # already canonical
    '5.png'      : '5.png',
    '6.png'                       : '6.png',   # already canonical
    '7.png'  : '7.png',
    
    '8.png'                    : '8.png',   # C-H plane → Fig 8
    '9.png'                       : '9.png',   # I+C_LMC temporal → Fig 9
}

for _src, _dst in _rename_map.items():
    if os.path.exists(_src) and _src != _dst:
        shutil.copy2(_src, _dst)
        print(f'  {_src} → {_dst}')
    elif _src == _dst:
        print(f'  {_src} — already canonical')
    else:
        print(f'  [MISSING] {_src}')

print('\nFinal figure set for submission:')
for _n in range(1, 10):
    _f = f'{_n}.png'
    _ok = os.path.exists(_f)
    print(f'  Fig {_n}: {_f}  {"OK" if _ok else "MISSING"}')
